In [8]:
%load_ext autoreload
%autoreload 2
import os, sys
from pathlib import Path

PROJECT_ROOT = Path("/home/liuxp/Documents/Projects/STHN_clean").resolve()

os.environ["CUDA_VISIBLE_DEVICES"] = "3"


import torch
import torch.nn.functional as F
import argparse
import numpy as np
from tqdm import tqdm



os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("cwd:", os.getcwd())
print("sys.path[0]:", sys.path[0])


from local_pipeline.model.network import STHN
import local_pipeline.datasets_4cor_img as datasets
from local_pipeline.utils import coords_grid, bilinear_sampler      
from local_pipeline.corr import CorrBlock




The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
cwd: /home/liuxp/Documents/Projects/STHN_clean
sys.path[0]: /home/liuxp/Documents/Projects/STHN_clean


In [9]:
# 验证 Notebook 环境是否能看到函数
from local_pipeline.utils import bilinear_sampler
print("Bilinear Sampler location:", bilinear_sampler)

# 验证 CorrBlock 内部是否能看到函数
from local_pipeline.corr import CorrBlock
# 检查 CorrBlock 的命名空间中是否包含 bilinear_sampler
import inspect
print("Does CorrBlock know bilinear_sampler?", 'bilinear_sampler' in dir(CorrBlock))

Bilinear Sampler location: <function bilinear_sampler at 0x79031efbcd30>
Does CorrBlock know bilinear_sampler? False


In [10]:
import torch
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu name:", torch.cuda.get_device_name(0))


torch version: 2.0.1+cu117
cuda available: True
gpu count: 1
gpu name: NVIDIA GeForce RTX 4090


# # 参数配置 (模拟 warm-brook-10 的设置)

In [11]:
args = argparse.Namespace()
args.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
args.gpuid = [0]
args.batch_size = 8  # 显存允许的话，越大跑得越快
args.num_workers = 4

# 数据集参数
args.dataset_name = "satellite_0_thermalmapping_135_train"       # 请确认这与你的文件夹名字一致
args.datasets_folder = "datasets"    # 指向你的软连接
args.resize_width = 256
args.crop_width = 256
args.database_size = 512             # 根据你的 datasets_4cor_img 逻辑设置

# 模型结构参数
args.arch = "IHN"
args.two_stages = False
args.use_cbam = False                # 如果测 SE 模型，请改为 True (如果复用了此开关)
args.lev0 = True
args.mixed_precision = False
args.fnet_cat = False                # IHN 默认为 False

# 其他必要的占位参数 (防止 dataloader 报错)
args.eval_model = None
args.perspective_max = 0
args.rotate_max = 0
args.resize_max = 0
args.G_contrast = "none"
args.augment = False                 # 验证模式必须关闭增强
args.val_positive_dist_threshold = 25
args.load_test_pairs = None
args.generate_test_pairs = False
args.prior_location_threshold = -1
args.weight = None
args.corr_level = 4

# ★★★ 模型权重路径 (请根据需要修改) ★★★
CHECKPOINT_PATH = "logs/local_he/20251229_024813_gnode_larger_512_mini_32/STHN.pth"
# CHECKPOINT_PATH = "checkpoints/still-cloud-13/model_10000.pth" 


# 3. 核心评估函数定义

In [12]:
@torch.no_grad()
def eval_feature_margin(fmap1, fmap2, flow_gt_full, num_samples=2048, neg_mode="local", local_radius=8):
    """ 计算特征的区分度 (Margin = 正样本相似度 - 负样本相似度) """
    device = fmap1.device
    B, C, Hq, Wq = fmap1.shape
    
    # 将 GT Flow 下采样到特征图分辨率 (通常是 1/4)
    flow_gt_q = F.interpolate(flow_gt_full, size=(Hq, Wq), mode="bilinear", align_corners=True) / 4.0
    
    # 归一化特征
    f1 = F.normalize(fmap1, dim=1)
    f2 = F.normalize(fmap2, dim=1)
    
    # 随机采样点
    N = min(num_samples, Hq * Wq)
    ys = torch.randint(0, Hq, (B, N), device=device)
    xs = torch.randint(0, Wq, (B, N), device=device)
    
    # 取出 f1 中的特征向量
    batch_idx = torch.arange(B, device=device).unsqueeze(1)
    f1_p = f1.permute(0, 2, 3, 1)[batch_idx, ys, xs, :] # (B, N, C)
    
    # 计算 f2 中对应的正样本位置: q+ = p + flow
    flow_vectors = flow_gt_q.permute(0, 2, 3, 1)[batch_idx, ys, xs, :] # (B, N, 2)
    qx = xs.float() + flow_vectors[..., 0]
    qy = ys.float() + flow_vectors[..., 1]
    
    # 双线性插值取出 f2 中的正样本特征
    coords_pos = torch.stack([qx, qy], dim=-1)
    # bilinear_sampler 期望坐标归一化到 [-1, 1]，或者直接处理像素坐标？
    # 注意：你的 utils.bilinear_sampler 可能需要像素坐标或归一化坐标
    # 这里为了通用性，假设它接受像素坐标，或者我们手动用 grid_sample
    # 既然你导入了 utils，我们假设 utils.bilinear_sampler 内部处理好了或者它是基于 grid_sample 的
    # 为了保险，这里用最原生的 grid_sample 逻辑重写一下采样部分，避免依赖 utils 的具体实现细节
    
    # --- 手动采样逻辑开始 ---
    # grid_sample 需要 [-1, 1]
    norm_qx = 2.0 * qx / (Wq - 1) - 1.0
    norm_qy = 2.0 * qy / (Hq - 1) - 1.0
    grid_pos = torch.stack([norm_qx, norm_qy], dim=-1).unsqueeze(2) # (B, N, 1, 2)
    f2_pos = F.grid_sample(f2, grid_pos, align_corners=True).squeeze(3).permute(0, 2, 1) # (B, N, C)
    # --- 手动采样逻辑结束 ---

    # 构建负样本位置
    if neg_mode == "random":
        nx = torch.randint(0, Wq, (B, N), device=device).float()
        ny = torch.randint(0, Hq, (B, N), device=device).float()
    else: # local hard negative
        offx = torch.randint(-local_radius, local_radius + 1, (B, N), device=device).float()
        offy = torch.randint(-local_radius, local_radius + 1, (B, N), device=device).float()
        nx = qx + offx
        ny = qy + offy
        
    norm_nx = 2.0 * nx / (Wq - 1) - 1.0
    norm_ny = 2.0 * ny / (Hq - 1) - 1.0
    grid_neg = torch.stack([norm_nx, norm_ny], dim=-1).unsqueeze(2)
    f2_neg = F.grid_sample(f2, grid_neg, align_corners=True).squeeze(3).permute(0, 2, 1)

    # 计算 Cosine Similarity
    sim_pos = (f1_p * f2_pos).sum(dim=-1)
    sim_neg = (f1_p * f2_neg).sum(dim=-1)
    
    # 过滤越界点
    valid = (qx >= 0) & (qx <= Wq-1) & (qy >= 0) & (qy <= Hq-1) & \
            (nx >= 0) & (nx <= Wq-1) & (ny >= 0) & (ny <= Hq-1)
            
    if valid.sum() == 0:
        return {"margin": np.nan}
        
    return {
        "pos_mean": sim_pos[valid].mean().item(),
        "neg_mean": sim_neg[valid].mean().item(),
        "margin": (sim_pos - sim_neg)[valid].mean().item()
    }


In [13]:
@torch.no_grad()
def eval_corr_sharpness(fmap1, fmap2, flow_gt_full, corr_radius=4):
    """ 计算相关性分布的尖锐程度 (Entropy) """
    device = fmap1.device
    B, _, Hq, Wq = fmap1.shape
    
    flow_gt_q = F.interpolate(flow_gt_full, size=(Hq, Wq), mode="bilinear", align_corners=True) / 4.0
    coords0 = coords_grid(B, Hq, Wq).to(device)
    coords_gt = coords0 + flow_gt_q
    
    # 构建 Correlation Volume
    corr_fn = CorrBlock(fmap1, fmap2, num_levels=2, radius=corr_radius)
    # 在 GT 位置采样 Correlation
    corr_feat = corr_fn(coords_gt) # (B, (2r+1)^2 * levels, Hq, Wq)
    
    # 简单起见，只看 Level 0 (最高分辨率层)
    dim0 = (2*corr_radius + 1)**2
    corr_lev0 = corr_feat[:, :dim0, ...] # (B, K, H, W)
    
    # 计算 Entropy
    # 随机采点评估
    K = 2048
    ys = torch.randint(0, Hq, (B, K), device=device)
    xs = torch.randint(0, Wq, (B, K), device=device)
    batch_idx = torch.arange(B, device=device).unsqueeze(1)
    
    vec = corr_lev0.permute(0, 2, 3, 1)[batch_idx, ys, xs, :] # (B, K, dim0)
    
    # Softmax 归一化
    probs = F.softmax(vec, dim=-1)
    entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=-1)
    
    peak_val = vec.max(dim=-1).values
    mean_val = vec.mean(dim=-1)
    
    return {
        "corr_peak": peak_val.mean().item(),
        "corr_gap": (peak_val - mean_val).mean().item(),
        "corr_entropy": entropy.mean().item()
    }

# 4. 主执行流程

In [14]:

print(f"🚀 Loading model: {CHECKPOINT_PATH}")
if not os.path.exists(CHECKPOINT_PATH):
    print("❌ Checkpoint file not found! Please check the path.")
else:
    # model = STHN(args)
    # state_dict = torch.load(CHECKPOINT_PATH, map_location='cpu')
    # model.netG.load_state_dict(state_dict)
    # model.netG.eval().cuda()


    model = STHN(args)
    checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')

    # 检查 checkpoint 是否包含 'netG' 这个键
    if 'netG' in checkpoint:
        # 既然保存时包了一层 'netG'，那我们就把这一层剥开，只取里面的内容
        model.netG.load_state_dict(checkpoint['netG'])
        print("✅ 成功从 checkpoint['netG'] 加载权重！")
    else:
        # 如果没有 'netG' 键，说明可能是一个纯粹的权重文件，尝试直接加载
        # 或者有时候权重被保存在 'state_dict' 或 'model' 键下，视保存代码而定
        try:
            model.netG.load_state_dict(checkpoint)
            print("✅ 成功直接加载权重！")
        except RuntimeError as e:
            print(f"❌ 加载失败，请检查键名。当前Checkpoint包含的键: {list(checkpoint.keys())}")
            raise e

    model.netG.eval().cuda()

    print("📂 Initializing Dataloader...")
    # 强制使用验证集
    val_loader = datasets.fetch_dataloader(args, split='val')

    metrics = {"pos_mean": [], "neg_mean": [], "margin": [], "corr_entropy": []}

    print("⚡ Starting Evaluation Loop...")
    for i, data in enumerate(tqdm(val_loader)):
        # datasets_4cor_img 返回: img2, img1, flow, H, query_utm, database_utm, index, pos_index
        img2, img1, flow_gt, _, _, _, _, _ = data
        
        img1 = img1.cuda()
        img2 = img2.cuda()
        flow_gt = flow_gt.cuda()

        if img2.shape[2:] != img1.shape[2:]:
            img2 = F.interpolate(img2, size=img1.shape[2:], mode='bilinear', align_corners=True)
        
        # ★★★ 关键适配：处理 fnet1 可能返回 tuple 的情况 ★★★
        out1 = model.netG.fnet1(img1)
        out2 = model.netG.fnet1(img2)
        
        # 自动识别提取特征
        fmap1 = out1[0] if isinstance(out1, (tuple, list)) else out1
        fmap2 = out2[0] if isinstance(out2, (tuple, list)) else out2
        
        fmap1 = fmap1.float()
        fmap2 = fmap2.float()
        
        # 计算指标
        m_res = eval_feature_margin(fmap1, fmap2, flow_gt)
        c_res = eval_corr_sharpness(fmap1, fmap2, flow_gt)
        
        if not np.isnan(m_res['margin']):
            metrics['pos_mean'].append(m_res['pos_mean'])
            metrics['neg_mean'].append(m_res['neg_mean'])
            metrics['margin'].append(m_res['margin'])
            metrics['corr_entropy'].append(c_res['corr_entropy'])
            
        # if i >= 50: break # 只测 50 个 batch 快速看结果

    print("\n" + "="*40)
    print(f"📊 Results for {os.path.basename(CHECKPOINT_PATH)}")
    print("="*40)
    print(f"Feature Margin (↑): {np.mean(metrics['margin']):.4f}")
    print(f"  - Pos Score:      {np.mean(metrics['pos_mean']):.4f}")
    print(f"  - Neg Score:      {np.mean(metrics['neg_mean']):.4f}")
    print(f"Corr Entropy   (↓): {np.mean(metrics['corr_entropy']):.4f}")
    print("="*40)

🚀 Loading model: logs/local_he/satellite_0_thermalmapping_135_train-2025-12-26_01-54-46-6a4b6bed-7eaa-42e9-b96f-8055bec7bbd2/STHN.pth


✅ 成功从 checkpoint['netG'] 加载权重！
📂 Initializing Dataloader...
Using online test pairs or generating test pairs. It is possible that different batch size can generate different test pairs.
⚡ Starting Evaluation Loop...


  0%|                                                    | 0/63 [00:00<?, ?it/s]/home/liuxp/.conda/envs/py310-sthn/lib/python3.10/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(
/home/liuxp/.conda/envs/py310-sthn/lib/python3.10/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resiz


📊 Results for STHN.pth
Feature Margin (↑): 0.0465
  - Pos Score:      0.1739
  - Neg Score:      0.1274
Corr Entropy   (↓): 0.8040
